# Day 12 - Pandas DataFrames: Basics + Advanced
### 90-Day Gen AI Engineer Roadmap
> **Dataset:** Titanic (Kaggle) - perfect for NaN handling, filtering, correlation

**Author:** Shaurab Kumar Jha  
**Date:** Day 11 of 90  
**Goal:** MNC-ready Python & Gen AI Engineer

## Table of Contents
1. [Setup & Dataset Download](#setup)
2. [DataFrame Creation](#creation)
3. [DataFrame Attributes](#attributes)
4. [Null & Duplicate Detection](#null)
5. [Rename Columns](#rename)
6. [Maths on DataFrame (axis)](#maths)
7. [Row/Column Selection](#selection)
8. [Filtering (Boolean Masking)](#filtering)
9. [Advanced Sorting](#sorting)
10. [Rank](#rank)
11. [set_index / reset_index](#index)
12. [unique / nunique / value_counts](#unique)
13. [NaN Handling](#nan)
14. [Duplicates](#duplicates)
15. [drop](#drop)
16. [apply](#apply)
17. [isin](#isin)
18. [corr - Correlation Matrix](#corr)
19. [nlargest / nsmallest](#nlargest)
20. [insert](#insert)
21. [copy](#copy)
22. [Interview Q&A Cheatsheet](#interview)


---
## 1.Setup & Dataset Download <a name='setup'></a>

In [46]:
# Install / import
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print(f'Pandas version: {pd.__version__}')
print(f'NumPy  version: {np.__version__}')

Pandas version: 2.2.3
NumPy  version: 2.3.3


In [47]:
# ── Download Titanic dataset from GitHub mirror ──────────────
# (No Kaggle account needed for this URL)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)
print('✅ Dataset loaded!')
print(f'Shape: {df.shape}  →  {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

✅ Dataset loaded!
Shape: (891, 12)  →  891 rows, 12 columns


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
## 2. DataFrame Creation <a name='creation'></a>

###  Theory
A **DataFrame** is a 2-dimensional labelled data structure - think of it as a spreadsheet or SQL table in Python.

**Three most common ways to create:**
| Method | Use when |
|--------|----------|
| `pd.DataFrame(list_of_dicts)` | Each dict = one row |
| `pd.DataFrame(dict_of_lists)` | Each key = one column |
| `pd.read_csv(path)` | Loading from file |


In [48]:
# ── From list of dicts ──────────────────────────────────────
# Each dict represents ONE ROW
data_list = [
    {'Name': 'Alice', 'Age': 24, 'Score': 88},
    {'Name': 'Bob',   'Age': 27, 'Score': 72},
    {'Name': 'Carol', 'Age': 22, 'Score': 95},
]
df_from_list = pd.DataFrame(data_list)
print('From list of dicts:')
df_from_list

From list of dicts:


,Name,Age,Score
0,Alice,24,88
1,Bob,27,72
2,Carol,22,95


In [49]:
# ── From dict of lists ──────────────────────────────────────
# Each key = ONE COLUMN, value = list of values for that column
data_dict = {
    'Name':  ['Alice', 'Bob', 'Carol'],
    'Age':   [24, 27, 22],
    'Score': [88, 72, 95],
}
df_from_dict = pd.DataFrame(data_dict)
print('From dict of lists:')
df_from_dict

From dict of lists:


,Name,Age,Score
0,Alice,24,88
1,Bob,27,72
2,Carol,22,95


In [50]:
# ── From CSV ────────────────────────────────────────────────
# We already loaded Titanic above. Useful params:
#   sep=','        → separator character (default comma)
#   header=0       → which row to use as column names
#   index_col='id' → which column to use as index
#   usecols=[...]  → load only certain columns
#   nrows=100      → load only first 100 rows
#   na_values=['-'] → treat these as NaN

# Example:
# df = pd.read_csv('titanic.csv', usecols=['Name','Age','Survived'], nrows=50)
print('CSV already loaded as df. Shape:', df.shape)

CSV already loaded as df. Shape: (891, 12)


---
## 3.DataFrame Attributes <a name='attributes'></a>

### Theory - All key attributes at a glance

| Attribute / Method | Returns | Use |
|---|---|---|
| `.shape` | tuple (rows, cols) | Quick size check |
| `.ndim` | int | Always 2 for DataFrame |
| `.size` | int | Total number of cells |
| `.dtypes` | Series | Data type of each column |
| `.columns` | Index | Column names |
| `.index` | Index | Row labels |
| `.values` | ndarray | Raw numpy array |
| `.head(n)` | DataFrame | First n rows (default 5) |
| `.tail(n)` | DataFrame | Last n rows |
| `.sample(n)` | DataFrame | n random rows |
| `.info()` | prints | Column info + dtypes + non-null count |
| `.describe()` | DataFrame | Stats for numeric cols |
| `.describe(include='all')` | DataFrame | Stats for ALL cols |


In [51]:
print('shape  →', df.shape)       # (891, 12)
print('ndim   →', df.ndim)        # 2
print('size   →', df.size)        # 891 * 12 = 10692
print()
print('columns →', df.columns.tolist())
print('index  →', df.index)       # RangeIndex(0..890)

shape  → (891, 12)
ndim   → 2
size   → 10692

columns → ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
index  → RangeIndex(start=0, stop=891, step=1)


In [52]:
print('dtypes:')
print(df.dtypes)
# int64   → whole numbers
# float64 → decimals (often means NaN present since int can't hold NaN)
# object  → strings / mixed

dtypes:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object


In [53]:
df.head(3)    # First 3 rows

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [54]:
df.tail(3)    # Last 3 rows

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


In [55]:
df.sample(3, random_state=42)    # Random 3 rows

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
709,710,1,3,"Moubarek, Master. Halim Gonios (""William George"")",male,NaN,1,1,2661,15.2458,NaN,C
439,440,0,2,"Kvillner, Mr. Johan Henrik Johannesson",male,31.0,0,0,C.A. 18723,10.5000,NaN,S
840,841,0,3,"Alhomaki, Mr. Ilmari Rudolf",male,20.0,0,0,SOTON/O2 3101287,7.9250,NaN,S


In [56]:
# info() → VERY useful: shows non-null count → tells you where NaNs are
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [57]:
# describe() → count, mean, std, min, quartiles, max
# Only numeric columns by default
df.describe().round(2)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.00,891.00,891.00,714.00,891.00,891.00,891.00
mean,446.00,0.38,2.31,29.70,0.52,0.38,32.20
std,257.35,0.49,0.84,14.53,1.10,0.81,49.69
min,1.00,0.00,1.00,0.42,0.00,0.00,0.00
25%,223.50,0.00,2.00,20.12,0.00,0.00,7.91
50%,446.00,0.00,3.00,28.00,0.00,0.00,14.45
75%,668.50,1.00,3.00,38.00,1.00,0.00,31.00
max,891.00,1.00,3.00,80.00,8.00,6.00,512.33


In [58]:
# describe(include='all') → also includes object columns
# For object cols: count, unique, top (most frequent), freq
df.describe(include='all')

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


---
## 4.Null & Duplicate Detection <a name='null'></a>

###  Theory
- `isnull()` → returns boolean DataFrame (True where NaN)
- `isnull().sum()` → count of NaNs per column ← **most used in EDA**
- `duplicated()` → True for rows that are exact duplicates
- `hasnans` → attribute on a Series, True if any NaN


In [59]:
# NaN count per column
null_counts = df.isnull().sum()
print('NaN counts:')
print(null_counts)
print()
print('Total NaNs in dataset:', df.isnull().sum().sum())

NaN counts:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Total NaNs in dataset: 866


In [60]:
# Percentage of NaN per column
nan_pct = (df.isnull().sum() / len(df) * 100).round(1)
print('NaN percentage per column:')
print(nan_pct[nan_pct > 0])    # only show cols with NaN

NaN percentage per column:
Age         19.9
Cabin       77.1
Embarked     0.2
dtype: float64


In [61]:
# hasnans - property on a Series
print('Age hasnans:', df['Age'].hasnans)
print('Survived hasnans:', df['Survived'].hasnans)

Age hasnans: True
Survived hasnans: False


In [62]:
# Duplicate rows
print('Duplicate rows count:', df.duplicated().sum())
# See the duplicate rows
df[df.duplicated()]    # empty if 0 duplicates

Duplicate rows count: 0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked


---
## 5.Rename <a name='rename'></a>

###  Theory
- `df.rename(columns={old: new})` → rename specific columns
- `df.rename(columns=str.lower)` → apply a function to all column names
- `inplace=True` → modify original; otherwise returns new df


In [63]:
# Rename specific columns using a dict
df_renamed = df.rename(columns={
    'Pclass':  'PassengerClass',
    'SibSp':   'Siblings_Spouses',
    'Parch':   'Parents_Children',
})
print('Renamed columns:', df_renamed.columns.tolist())

Renamed columns: ['PassengerId', 'Survived', 'PassengerClass', 'Name', 'Sex', 'Age', 'Siblings_Spouses', 'Parents_Children', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [64]:
# Rename using a function - lowercase all column names
df_lower = df.rename(columns=str.lower)
print('Lowercase cols:', df_lower.columns.tolist())

Lowercase cols: ['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


---
## 6.Maths on DataFrame (axis param) <a name='maths'></a>

###  Theory - axis explained
```
axis=0  →  operate DOWN the rows (column-wise result)
axis=1  →  operate ACROSS the columns (row-wise result)

Think of it as: which axis do you COLLAPSE?
axis=0 → collapse rows → one value per column
axis=1 → collapse cols → one value per row
```

| Operation | axis=0 (default) | axis=1 |
|-----------|-----------------|--------|
| `.sum()` | sum of each column | sum of each row |
| `.mean()` | mean of each column | mean of each row |
| `.var()` | variance of each column | variance of each row |


In [65]:
# Select only numeric columns
num_df = df.select_dtypes(include='number')
print('Numeric columns:', num_df.columns.tolist())

Numeric columns: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']


In [66]:
# Column-wise operations (axis=0) - default
print('SUM (axis=0):')
print(num_df.sum(axis=0))
print()
print('MEAN (axis=0):')
print(num_df.mean(axis=0).round(2))

SUM (axis=0):
PassengerId    397386.0000
Survived          342.0000
Pclass           2057.0000
Age             21205.1700
SibSp             466.0000
Parch             340.0000
Fare            28693.9493
dtype: float64

MEAN (axis=0):
PassengerId    446.00
Survived         0.38
Pclass           2.31
Age             29.70
SibSp            0.52
Parch            0.38
Fare            32.20
dtype: float64


In [67]:
# Row-wise operations (axis=1)
# Useful: total numeric value per passenger
print('SUM (axis=1) - first 5 rows:')
print(num_df.sum(axis=1).head())
print()
print('VAR (axis=0):')
print(num_df.var(axis=0).round(2))

SUM (axis=1) - first 5 rows:
0     34.2500
1    114.2833
2     40.9250
3     95.1000
4     51.0500
dtype: float64

VAR (axis=0):
PassengerId    66231.00
Survived           0.24
Pclass             0.70
Age              211.02
SibSp              1.22
Parch              0.65
Fare            2469.44
dtype: float64


---
## 7.Row / Column Selection <a name='selection'></a>

### Theory - loc vs iloc
| | `loc` | `iloc` |
|-|-------|--------|
| Based on | **Labels** (index names / column names) | **Integer positions** (0, 1, 2...) |
| Row slice | Inclusive on both ends | Exclusive on right (like Python) |
| Best for | Named indexes, label-based filtering | Position-based slicing |

**Memory trick:** `loc` → **l**abels; `iloc` → **i**nteger


In [68]:
# Single column → Series
ages = df['Age']
print(type(ages))    # <class 'pandas.core.series.Series'>
ages.head()

<class 'pandas.core.series.Series'>


0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
Name: Age, dtype: float64

In [69]:
# Multiple columns → DataFrame
subset = df[['Name', 'Survived', 'Age', 'Pclass']]
print(type(subset))  # DataFrame
subset.head()

<class 'pandas.core.frame.DataFrame'>


,Name,Survived,Age,Pclass
0,"Braund, Mr. Owen Harris",0,22.0,3
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1
2,"Heikkinen, Miss. Laina",1,26.0,3
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1
4,"Allen, Mr. William Henry",0,35.0,3


In [70]:
# ── iloc - Integer position based ───────────────────────────
print('iloc[0]  - first row:')
print(df.iloc[0])    # returns Series

iloc[0]  - first row:
PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
SibSp                                1
Parch                                0
Ticket                       A/5 21171
Fare                              7.25
Cabin                              NaN
Embarked                             S
Name: 0, dtype: object


In [71]:
print('iloc[0:3] - rows 0, 1, 2:')
df.iloc[0:3]

iloc[0:3] - rows 0, 1, 2:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [72]:
print('iloc[0:3, 0:4] - first 3 rows, first 4 columns:')
df.iloc[0:3, 0:4]

iloc[0:3, 0:4] - first 3 rows, first 4 columns:


,PassengerId,Survived,Pclass,Name
0,1,0,3,"Braund, Mr. Owen Harris"
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,3,1,3,"Heikkinen, Miss. Laina"


In [73]:
# Specific rows and columns using lists
print('iloc[[0,2,4], [1,3,5]] - specific rows & cols:')
df.iloc[[0, 2, 4], [1, 3, 5]]

iloc[[0,2,4], [1,3,5]] - specific rows & cols:


,Survived,Name,Age
0,0,"Braund, Mr. Owen Harris",22.0
2,1,"Heikkinen, Miss. Laina",26.0
4,0,"Allen, Mr. William Henry",35.0


In [74]:
# ── loc - Label based ───────────────────────────────────────
print('loc[0, "Name"] - single cell:')
print(df.loc[0, 'Name'])

loc[0, "Name"] - single cell:
Braund, Mr. Owen Harris


In [75]:
# loc with row range and column list
# Note: loc is INCLUSIVE on both ends (0:2 includes row 2)
df.loc[0:2, ['Name', 'Age', 'Survived', 'Pclass']]

,Name,Age,Survived,Pclass
0,"Braund, Mr. Owen Harris",22.0,0,3
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,1
2,"Heikkinen, Miss. Laina",26.0,1,3


In [76]:
# loc with boolean condition (this is what filtering uses internally)
df.loc[df['Survived'] == 1, ['Name', 'Age']].head()

,Name,Age
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0
2,"Heikkinen, Miss. Laina",26.0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0


---
## 8.Filtering (Boolean Masking) <a name='filtering'></a>

### Theory - Boolean Masking
```python
# A boolean mask is a Series of True/False
mask = df['Age'] > 30        # returns Series of bool
df[mask]                     # keeps only True rows

# AND  → &    (not 'and')
# OR   → |    (not 'or')
# NOT  → ~    (not 'not')
# ALWAYS wrap each condition in parentheses!
```

**Common mistake:** `df[df['Age'] > 30 and df['Pclass'] == 1]` → ❌ Error
**Correct:** `df[(df['Age'] > 30) & (df['Pclass'] == 1)]` → ✅


In [77]:
# Single condition
survivors = df[df['Survived'] == 1]
print(f'Total survivors: {len(survivors)} out of {len(df)}')
survivors[['Name', 'Age', 'Pclass']].head()

Total survivors: 342 out of 891


,Name,Age,Pclass
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1
2,"Heikkinen, Miss. Laina",26.0,3
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,3
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,2


In [78]:
# AND condition - both must be True
first_class_survivors = df[(df['Survived'] == 1) & (df['Pclass'] == 1)]
print(f'1st class survivors: {len(first_class_survivors)}')
first_class_survivors[['Name', 'Age', 'Fare']].head()

1st class survivors: 136


,Name,Age,Fare
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,53.1000
11,"Bonnell, Miss. Elizabeth",58.0,26.5500
23,"Sloper, Mr. William Thompson",28.0,35.5000
31,"Spencer, Mrs. William Augustus (Marie Eugenie)",NaN,146.5208


In [79]:
# OR condition - at least one must be True
young_or_rich = df[(df['Age'] < 18) | (df['Pclass'] == 1)]
print(f'Passengers under 18 OR in 1st class: {len(young_or_rich)}')

Passengers under 18 OR in 1st class: 317


In [80]:
# NOT condition using ~
non_male = df[~(df['Sex'] == 'male')]
print(f'Non-male passengers: {len(non_male)}')

Non-male passengers: 314


In [81]:
# str.contains - search within string columns
# na=False → treat NaN as False (don't crash)
miss_passengers = df[df['Name'].str.contains('Miss', na=False)]
print(f'Passengers with "Miss" in name: {len(miss_passengers)}')
miss_passengers[['Name', 'Age', 'Survived']].head()

Passengers with "Miss" in name: 182


,Name,Age,Survived
2,"Heikkinen, Miss. Laina",26.0,1
10,"Sandstrom, Miss. Marguerite Rut",4.0,1
11,"Bonnell, Miss. Elizabeth",58.0,1
14,"Vestrom, Miss. Hulda Amanda Adolfina",14.0,0
22,"McGowan, Miss. Anna ""Annie""",15.0,1


In [82]:
# Adding a new column
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1   # +1 for passenger themselves
print('FamilySize column added!')
df[['Name', 'SibSp', 'Parch', 'FamilySize']].head()

FamilySize column added!


,Name,SibSp,Parch,FamilySize
0,"Braund, Mr. Owen Harris",1,0,2
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,2
2,"Heikkinen, Miss. Laina",0,0,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,2
4,"Allen, Mr. William Henry",0,0,1


In [83]:
# astype - convert column dtype
df['Survived_bool'] = df['Survived'].astype(bool)
df['Pclass_str']    = df['Pclass'].astype(str)
print('Survived bool:', df['Survived_bool'].dtype)
print('Pclass str:   ', df['Pclass_str'].dtype)

Survived bool: bool
Pclass str:    object


In [84]:
# value_counts - frequency of each unique value
print('Pclass value_counts:')
print(df['Pclass'].value_counts())
print()
print('Proportions (normalize=True):')
print(df['Pclass'].value_counts(normalize=True).round(3))

Pclass value_counts:
Pclass
3    491
1    216
2    184
Name: count, dtype: int64

Proportions (normalize=True):
Pclass
3    0.551
1    0.242
2    0.207
Name: proportion, dtype: float64


In [85]:
# sort_values - basic
print('Youngest passengers:')
df[['Name', 'Age']].sort_values('Age').head(5)

Youngest passengers:


,Name,Age
803,"Thomas, Master. Assad Alexander",0.42
755,"Hamalainen, Master. Viljo",0.67
644,"Baclini, Miss. Eugenie",0.75
469,"Baclini, Miss. Helene Barbara",0.75
78,"Caldwell, Master. Alden Gates",0.83


In [86]:
# sort_values - multiple columns
print('Sort by Pclass (asc) then Fare (desc):')
df[['Name', 'Pclass', 'Fare']].sort_values(
    ['Pclass', 'Fare'],
    ascending=[True, False]
).head(6)

Sort by Pclass (asc) then Fare (desc):


,Name,Pclass,Fare
258,"Ward, Miss. Anna",1,512.3292
679,"Cardeza, Mr. Thomas Drake Martinez",1,512.3292
737,"Lesurer, Mr. Gustave J",1,512.3292
27,"Fortune, Mr. Charles Alexander",1,263.0000
88,"Fortune, Miss. Mabel Helen",1,263.0000
341,"Fortune, Miss. Alice Elizabeth",1,263.0000


---
## 9. Advanced Sorting <a name='sorting'></a>

###  Theory - All sort_values parameters
```python
df.sort_values(
    by='Age',           # column name or list of column names
    ascending=True,     # True=asc, False=desc. List for multiple cols
    na_position='last', # 'first' or 'last' - where to put NaN
    inplace=False,      # True = modify original, False = return new df
    ignore_index=False, # True = reset index after sorting
    key=None,           # function to apply to values before sorting
)
```


In [87]:
# na_position - where NaN goes in sorted output
print('NaN at LAST (default):')
print(df[['Name', 'Age']].sort_values('Age', na_position='last').tail(3))
print()
print('NaN at FIRST:')
print(df[['Name', 'Age']].sort_values('Age', na_position='first').head(3))

NaN at LAST (default):
                                         Name  Age
868               van Melkebeke, Mr. Philemon  NaN
878                        Laleff, Mr. Kristo  NaN
888  Johnston, Miss. Catherine Helen "Carrie"  NaN

NaN at FIRST:
                            Name  Age
5               Moran, Mr. James  NaN
17  Williams, Mr. Charles Eugene  NaN
19       Masselmani, Mrs. Fatima  NaN


In [88]:
# inplace=True - modifies the DataFrame directly
# WARNING: You can't undo inplace! Always work on a copy when experimenting
df_copy = df.copy()
df_copy.sort_values('Fare', ascending=False, inplace=True)
print('After inplace sort by Fare (desc):')
df_copy[['Name', 'Fare']].head(4)

After inplace sort by Fare (desc):


,Name,Fare
679,"Cardeza, Mr. Thomas Drake Martinez",512.3292
258,"Ward, Miss. Anna",512.3292
737,"Lesurer, Mr. Gustave J",512.3292
88,"Fortune, Miss. Mabel Helen",263.0000


In [89]:
# sort_index - sort by index label (useful after set_index or after filtering)
df_shuffled = df.sample(frac=1, random_state=1)  # shuffle rows
print('Shuffled index (first 6):', df_shuffled.index.tolist()[:6])

df_sorted_idx = df_shuffled.sort_index()
print('After sort_index (first 6):', df_sorted_idx.index.tolist()[:6])

Shuffled index (first 6): [862, 223, 84, 680, 535, 623]
After sort_index (first 6): [0, 1, 2, 3, 4, 5]


---
## 10. Rank <a name='rank'></a>

###  Theory
`rank()` assigns a rank to each value. Useful for percentile calculations.

| method | behavior when tied |
|--------|-------------------|
| `average` | average of ranks (default) |
| `min` | lowest rank |
| `max` | highest rank |
| `first` | rank in order of appearance |
| `dense` | no gaps in ranks |


In [90]:
df['Fare_rank'] = df['Fare'].rank(ascending=False)  # highest fare = rank 1
df[['Name', 'Fare', 'Fare_rank']].sort_values('Fare_rank').head(5)

,Name,Fare,Fare_rank
679,"Cardeza, Mr. Thomas Drake Martinez",512.3292,2.0
737,"Lesurer, Mr. Gustave J",512.3292,2.0
258,"Ward, Miss. Anna",512.3292,2.0
341,"Fortune, Miss. Alice Elizabeth",263.0000,5.5
88,"Fortune, Miss. Mabel Helen",263.0000,5.5


In [91]:
# dense rank - no gaps (1, 2, 3... even when tied)
df['Age_dense_rank'] = df['Age'].rank(method='dense', na_option='keep')
# na_option='keep' → NaN stays NaN
# na_option='top'  → NaN gets smallest rank
# na_option='bottom'→ NaN gets largest rank
df[['Name', 'Age', 'Age_dense_rank']].dropna().head(6)

,Name,Age,Age_dense_rank
0,"Braund, Mr. Owen Harris",22.0,29.0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,52.0
2,"Heikkinen, Miss. Laina",26.0,35.0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,48.0
4,"Allen, Mr. William Henry",35.0,48.0
6,"McCarthy, Mr. Timothy J",54.0,70.0


---
## 11.set_index / reset_index <a name='index'></a>

###  Theory
- By default, DataFrame has RangeIndex (0, 1, 2, 3...)
- `set_index('col')` → use a column as the index (faster label-based access)
- `reset_index()` → bring index back as a column, restore RangeIndex
- After `set_index`, use `.loc[label]` not `.iloc[position]`


In [92]:
df_indexed = df.set_index('PassengerId')
print('After set_index, index:', df_indexed.index[:5].tolist())
df_indexed.head(3)

After set_index, index: [1, 2, 3, 4, 5]


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,Survived_bool,Pclass_str,Fare_rank,Age_dense_rank
PassengerId,,,,,,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,False,3,815.0,29.0
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,True,1,103.0,52.0
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,True,3,659.5,35.0


In [93]:
# Access by PassengerId label
print('Passenger with ID 5:')
df_indexed.loc[5, ['Name', 'Age', 'Survived']]

Passenger with ID 5:


Name        Allen, Mr. William Henry
Age                             35.0
Survived                           0
Name: 5, dtype: object

In [94]:
# reset_index - PassengerId comes back as a column
df_reset = df_indexed.reset_index()
print('After reset_index columns:', df_reset.columns.tolist()[:4])
df_reset.head(2)

After reset_index columns: ['PassengerId', 'Survived', 'Pclass', 'Name']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,Survived_bool,Pclass_str,Fare_rank,Age_dense_rank
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,False,3,815.0,29.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,True,1,103.0,52.0


---
## 12. unique / nunique / value_counts <a name='unique'></a>

###  Theory
| Method | Returns | Use |
|--------|---------|-----|
| `.unique()` | array of unique values | What distinct values exist? |
| `.nunique()` | int count of unique values | How many distinct values? |
| `.value_counts()` | Series (value → count) | How often does each value appear? |


In [95]:
print('Sex unique values:    ', df['Sex'].unique())
print('Pclass unique values: ', df['Pclass'].unique())
print('Embarked nunique:     ', df['Embarked'].nunique())
print('Age nunique:          ', df['Age'].nunique())

Sex unique values:     ['male' 'female']
Pclass unique values:  [3 1 2]
Embarked nunique:      3
Age nunique:           88


In [96]:
# value_counts - sorted by frequency (descending)
print('Embarked value_counts:')
print(df['Embarked'].value_counts())
print()
# dropna=False → include NaN in count
print('Embarked value_counts (with NaN):')
print(df['Embarked'].value_counts(dropna=False))

Embarked value_counts:
Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

Embarked value_counts (with NaN):
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64


In [97]:
# Survival rate per Pclass - groupby preview
print('Survival rate per class:')
print(df.groupby('Pclass')['Survived'].mean().round(2))

Survival rate per class:
Pclass
1    0.63
2    0.47
3    0.24
Name: Survived, dtype: float64


---
## 13. NaN Handling <a name='nan'></a>

###  Theory - Full NaN strategy

**Detection:**
- `isnull()` / `isna()` → both same thing, True where NaN
- `notnull()` / `notna()` → opposite
- `hasnans` → attribute on Series

**Removal:**
- `dropna(how='any')` → drop row if ANY column has NaN (default)
- `dropna(how='all')` → drop row only if ALL columns are NaN
- `dropna(subset=['Age'])` → only check specific columns
- `dropna(thresh=n)` → keep rows with at least n non-NaN values

**Imputation (filling):**
| Strategy | Best for |
|----------|----------|
| `fillna(mean)` | Numeric, roughly normal distribution |
| `fillna(median)` | Numeric, skewed data / outliers |
| `fillna(mode)` | Categorical columns |
| `fillna('Unknown')` | String columns |
| `ffill()` | Time series (use previous value) |
| `bfill()` | Time series (use next value) |


In [98]:
print('NaN per column:')
print(df.isnull().sum())
print()
# Age: 177 missing → impute with median
# Cabin: 687 missing (77%!) → too many, create 'Has_Cabin' feature or drop
# Embarked: 2 missing → impute with mode

NaN per column:
PassengerId         0
Survived            0
Pclass              0
Name                0
Sex                 0
Age               177
SibSp               0
Parch               0
Ticket              0
Fare                0
Cabin             687
Embarked            2
FamilySize          0
Survived_bool       0
Pclass_str          0
Fare_rank           0
Age_dense_rank    177
dtype: int64



In [99]:
# dropna examples
df_drop_any = df.dropna()                               # any NaN in row → drop row
df_drop_sub = df.dropna(subset=['Age', 'Embarked'])    # only check Age & Embarked
df_drop_thr = df.dropna(thresh=10)                     # keep if ≥10 non-NaN values

print(f'Original:              {len(df)} rows')
print(f'dropna(any):           {len(df_drop_any)} rows')
print(f'dropna(subset=[Age,E]): {len(df_drop_sub)} rows')
print(f'dropna(thresh=10):     {len(df_drop_thr)} rows')

Original:              891 rows
dropna(any):           183 rows
dropna(subset=[Age,E]): 712 rows
dropna(thresh=10):     891 rows


In [100]:
# fillna - create a clean copy for ML
df_clean = df.copy()

# Age: fill with median (robust to outliers)
age_median = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(age_median)
print(f'Age median used for filling: {age_median}')
print(f'Age NaN after fill: {df_clean["Age"].isnull().sum()}')

Age median used for filling: 28.0
Age NaN after fill: 0


In [101]:
# Embarked: fill with mode (most frequent)
embarked_mode = df_clean['Embarked'].mode()[0]    # mode() returns Series, [0] gets value
df_clean['Embarked'] = df_clean['Embarked'].fillna(embarked_mode)
print(f'Embarked mode: {embarked_mode}')
print(f'Embarked NaN after fill: {df_clean["Embarked"].isnull().sum()}')

Embarked mode: S
Embarked NaN after fill: 0


In [102]:
# Cabin: Create binary feature instead of filling (too many NaNs)
df_clean['Has_Cabin'] = df_clean['Cabin'].notnull().astype(int)
print('Has_Cabin distribution:')
print(df_clean['Has_Cabin'].value_counts())
# Insight: Having cabin info correlates with 1st class → higher survival

Has_Cabin distribution:
Has_Cabin
0    687
1    204
Name: count, dtype: int64


In [103]:
# ffill / bfill (forward fill / backward fill)
# Mostly useful for time series
sample_ages = df[['Age']].head(10).copy()
print('Original (with NaN):')
print(sample_ages.T)
print('\nAfter ffill:')
print(sample_ages.ffill().T)

Original (with NaN):
        0     1     2     3     4   5     6    7     8     9
Age  22.0  38.0  26.0  35.0  35.0 NaN  54.0  2.0  27.0  14.0

After ffill:
        0     1     2     3     4     5     6    7     8     9
Age  22.0  38.0  26.0  35.0  35.0  35.0  54.0  2.0  27.0  14.0


---
## 14. Duplicates <a name='duplicates'></a>

###  Theory
- `duplicated()` → returns boolean Series (True = duplicate row)
  - `keep='first'` → mark all duplicates EXCEPT first occurrence as True
  - `keep='last'` → mark all duplicates EXCEPT last occurrence as True
  - `keep=False` → mark ALL duplicates as True
- `drop_duplicates()` → remove duplicate rows
  - `subset=['col']` → only consider certain columns when checking duplicates


In [104]:
# Create artificial duplicates for demo
df_dup = pd.concat([df, df.iloc[[0, 1, 2]]], ignore_index=True)
print(f'With artificial duplicates: {len(df_dup)} rows')
print(f'Duplicate count: {df_dup.duplicated().sum()}')

With artificial duplicates: 894 rows
Duplicate count: 3


In [105]:
# See which rows are duplicated
df_dup[df_dup.duplicated(keep=False)][['Name', 'Age']].sort_values('Name')

,Name,Age
0,"Braund, Mr. Owen Harris",22.0
891,"Braund, Mr. Owen Harris",22.0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0
892,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0
2,"Heikkinen, Miss. Laina",26.0
893,"Heikkinen, Miss. Laina",26.0


In [106]:
# drop_duplicates
df_no_dup = df_dup.drop_duplicates()
print(f'After drop_duplicates(): {len(df_no_dup)} rows')

# subset - check duplicates based only on specific columns
df_sub_dup = df_dup.drop_duplicates(subset=['Name', 'Pclass'])
print(f'After drop_duplicates(subset): {len(df_sub_dup)} rows')

After drop_duplicates(): 891 rows
After drop_duplicates(subset): 891 rows


---
## 15. drop <a name='drop'></a>

###  Theory
```python
df.drop(columns=['col1', 'col2'])   # drop columns
df.drop(index=[0, 2, 5])            # drop rows by index label
```


In [107]:
# Drop columns
df_drop_cols = df.drop(columns=['Ticket', 'Cabin', 'Fare_rank', 'Age_dense_rank'])
print('After dropping cols:', df_drop_cols.columns.tolist())

After dropping cols: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'Survived_bool', 'Pclass_str']


In [108]:
# Drop rows by index
df_drop_rows = df.drop(index=[0, 1, 2])
print(f'After drop(index=[0,1,2]): {len(df_drop_rows)} rows')
print('First index now:', df_drop_rows.index[0])

After drop(index=[0,1,2]): 888 rows
First index now: 3


---
## 16. apply <a name='apply'></a>

###  Theory
- `apply()` = run a function on each element/row/column
- On a **Series**: applies function to each element
- On a **DataFrame** with `axis=0`: applies to each column (Series)
- On a **DataFrame** with `axis=1`: applies to each row (Series)

**When to use apply vs vectorized ops?**
- Vectorized (`.str`, `.dt`, `+`, `*`) → faster, prefer this
- `apply` → for complex custom logic that can't be vectorized


In [109]:
# apply with lambda on a Series
df_clean['Age_group'] = df_clean['Age'].apply(
    lambda x: 'Child' if x < 18 else ('Senior' if x >= 60 else 'Adult')
)
print('Age group distribution:')
print(df_clean['Age_group'].value_counts())

Age group distribution:
Age_group
Adult     752
Child     113
Senior     26
Name: count, dtype: int64


In [110]:
# apply with a named function
def fare_category(fare):
    if fare < 10:   return 'Budget'
    elif fare < 50: return 'Standard'
    else:            return 'Premium'

df_clean['Fare_category'] = df_clean['Fare'].apply(fare_category)
df_clean[['Name', 'Fare', 'Fare_category']].head(6)

,Name,Fare,Fare_category
0,"Braund, Mr. Owen Harris",7.2500,Budget
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",71.2833,Premium
2,"Heikkinen, Miss. Laina",7.9250,Budget
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",53.1000,Premium
4,"Allen, Mr. William Henry",8.0500,Budget
5,"Moran, Mr. James",8.4583,Budget


In [111]:
# apply on DataFrame (row-wise, axis=1)
def is_solo_traveler(row):
    return 1 if (row['SibSp'] == 0 and row['Parch'] == 0) else 0

df_clean['Is_Alone'] = df_clean.apply(is_solo_traveler, axis=1)
print('Alone vs with family:')
print(df_clean['Is_Alone'].value_counts())
print()
print('Survival rate - alone vs with family:')
print(df_clean.groupby('Is_Alone')['Survived'].mean().round(3))

Alone vs with family:
Is_Alone
1    537
0    354
Name: count, dtype: int64

Survival rate - alone vs with family:
Is_Alone
0    0.506
1    0.304
Name: Survived, dtype: float64


---
## 17. isin <a name='isin'></a>

###  Theory
- `isin([val1, val2])` → True where value is in the list
- Equivalent to SQL's `WHERE col IN (val1, val2)`
- Use `~isin()` for NOT IN


In [112]:
# Passengers who embarked at Cherbourg or Southampton
cs_passengers = df[df['Embarked'].isin(['C', 'S'])]
print(f'C or S passengers: {len(cs_passengers)}')

# First or second class only
upper_class = df[df['Pclass'].isin([1, 2])]
print(f'1st & 2nd class: {len(upper_class)}')

# NOT in 3rd class (using ~)
not_3rd = df[~df['Pclass'].isin([3])]
print(f'Not 3rd class: {len(not_3rd)}')

C or S passengers: 812
1st & 2nd class: 400
Not 3rd class: 400


---
## 18. corr - Correlation Matrix <a name='corr'></a>

###  Theory - Pearson Correlation Coefficient
- Ranges from **-1 to +1**
- `+1` → perfect positive correlation (as X goes up, Y goes up)
- `-1` → perfect negative correlation (as X goes up, Y goes down)
- `0` → no linear relationship
- Rule of thumb: |r| > 0.7 = strong, 0.4-0.7 = moderate, < 0.4 = weak

**For Titanic - expected correlations:**
- `Pclass ↔ Survived`: negative (higher class number = worse class = lower survival)
- `Fare ↔ Survived`: positive (paid more = 1st class = higher survival)
- `Fare ↔ Pclass`: negative (1st class → higher fare)


In [113]:
# Correlation matrix - numeric columns only
numeric_clean = df_clean.select_dtypes(include='number')
corr_matrix = numeric_clean.corr()
print('Correlation Matrix:')
corr_matrix.round(2)

Correlation Matrix:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,Fare_rank,Age_dense_rank,Has_Cabin,Is_Alone
PassengerId,1.00,-0.01,-0.04,0.03,-0.06,-0.00,0.01,-0.04,0.01,0.04,0.02,0.06
Survived,-0.01,1.00,-0.34,-0.06,-0.04,0.08,0.26,0.02,-0.32,-0.07,0.32,-0.20
Pclass,-0.04,-0.34,1.00,-0.34,0.08,0.02,-0.55,0.07,0.70,-0.37,-0.73,0.14
Age,0.03,-0.06,-0.34,1.00,-0.23,-0.17,0.10,-0.25,-0.11,1.00,0.24,0.17
SibSp,-0.06,-0.04,0.08,-0.23,1.00,0.41,0.16,0.89,-0.36,-0.30,-0.04,-0.58
Parch,-0.00,0.08,0.02,-0.17,0.41,1.00,0.22,0.78,-0.37,-0.18,0.04,-0.58
Fare,0.01,0.26,-0.55,0.10,0.16,0.22,1.00,0.22,-0.64,0.10,0.48,-0.27
FamilySize,-0.04,0.02,0.07,-0.25,0.89,0.78,0.22,1.00,-0.43,-0.29,-0.01,-0.69
Fare_rank,0.01,-0.32,0.70,-0.11,-0.36,-0.37,-0.64,-0.43,1.00,-0.12,-0.54,0.53
Age_dense_rank,0.04,-0.07,-0.37,1.00,-0.30,-0.18,0.10,-0.29,-0.12,1.00,0.25,0.18


In [114]:
# Most important: correlation with Survived
print('Correlation with Survived (sorted):')
print(corr_matrix['Survived'].sort_values(ascending=False).round(3))
print()
print('Insights:')
print('  • Pclass is NEGATIVE → lower class = lower survival')
print('  • Fare is POSITIVE   → higher fare = higher survival')
print('  • Age is NEGATIVE    → older = slightly lower survival')

Correlation with Survived (sorted):
Survived          1.000
Has_Cabin         0.317
Fare              0.257
Parch             0.082
FamilySize        0.017
PassengerId      -0.005
SibSp            -0.035
Age              -0.065
Age_dense_rank   -0.071
Is_Alone         -0.203
Fare_rank        -0.324
Pclass           -0.338
Name: Survived, dtype: float64

Insights:
  • Pclass is NEGATIVE → lower class = lower survival
  • Fare is POSITIVE   → higher fare = higher survival
  • Age is NEGATIVE    → older = slightly lower survival


In [115]:
# Visualize correlation as heatmap
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt='.2f',
        cmap='RdBu_r',
        center=0,
        square=True,
        linewidths=0.5
    )
    plt.title('Titanic - Correlation Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib/seaborn not available. Run: pip install matplotlib seaborn')

matplotlib/seaborn not available. Run: pip install matplotlib seaborn


---
## 19. nlargest / nsmallest <a name='nlargest'></a>

###  Theory
- `nlargest(n, 'col')` → n rows with LARGEST values in col
- `nsmallest(n, 'col')` → n rows with SMALLEST values in col
- Faster than `sort_values().head(n)` for large datasets


In [116]:
# Top 5 highest fares
print('Top 5 passengers by Fare:')
df[['Name', 'Fare', 'Pclass', 'Survived']].nlargest(5, 'Fare')

Top 5 passengers by Fare:


,Name,Fare,Pclass,Survived
258,"Ward, Miss. Anna",512.3292,1,1
679,"Cardeza, Mr. Thomas Drake Martinez",512.3292,1,1
737,"Lesurer, Mr. Gustave J",512.3292,1,1
27,"Fortune, Mr. Charles Alexander",263.0000,1,0
88,"Fortune, Miss. Mabel Helen",263.0000,1,1


In [117]:
# 5 youngest passengers
print('5 youngest passengers:')
df_clean[['Name', 'Age', 'Survived']].nsmallest(5, 'Age')

5 youngest passengers:


,Name,Age,Survived
803,"Thomas, Master. Assad Alexander",0.42,1
755,"Hamalainen, Master. Viljo",0.67,1
469,"Baclini, Miss. Helene Barbara",0.75,1
644,"Baclini, Miss. Eugenie",0.75,1
78,"Caldwell, Master. Alden Gates",0.83,1


In [118]:
# Richest survivors
surv_df = df_clean[df_clean['Survived'] == 1]
print('Top 3 richest survivors (by fare):')
surv_df[['Name', 'Fare', 'Pclass']].nlargest(3, 'Fare')

Top 3 richest survivors (by fare):


,Name,Fare,Pclass
258,"Ward, Miss. Anna",512.3292,1
679,"Cardeza, Mr. Thomas Drake Martinez",512.3292,1
737,"Lesurer, Mr. Gustave J",512.3292,1


---
## 20.insert <a name='insert'></a>

###  Theory
- `insert(loc, column, value)` → insert a new column at a specific position
- Unlike assignment (`df['col'] = ...`) which always adds at the end
- `loc` → integer position (0 = leftmost)
- `allow_duplicates=False` → raises error if column name already exists


In [119]:
df_ins = df_clean.copy()
print('Columns before insert:', df_ins.columns.tolist()[:5])

# Insert 'Is_Alone' at position 1 (right after PassengerId)
df_ins.insert(
    loc=1,
    column='Is_Solo',
    value=(df_ins['FamilySize'] == 1).astype(int)
)
print('\nColumns after insert:', df_ins.columns.tolist()[:6])
df_ins[['PassengerId', 'Is_Solo', 'FamilySize', 'Name']].head(4)

Columns before insert: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex']

Columns after insert: ['PassengerId', 'Is_Solo', 'Survived', 'Pclass', 'Name', 'Sex']


,PassengerId,Is_Solo,FamilySize,Name
0,1,0,2,"Braund, Mr. Owen Harris"
1,2,0,2,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,3,1,1,"Heikkinen, Miss. Laina"
3,4,0,2,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"


---
## 21. copy <a name='copy'></a>

###  Theory - The View Problem
This is one of the most common bugs in Pandas!

```python
df2 = df        # NOT a copy! df2 and df share the same memory
df2 = df.copy() # TRUE copy - independent from df

# Slice may be a VIEW (not a copy):
slice = df[df['Age'] > 30]      # might be a view
slice['NewCol'] = 1             # SettingWithCopyWarning!

# Safe way:
slice = df[df['Age'] > 30].copy()  # force a copy
slice['NewCol'] = 1             # safe, no warning
```

**Rule: Always use `.copy()` when you plan to modify a subset!**


In [120]:
# Demonstrate copy vs reference
df_original = df[['Name', 'Age']].head(5)

# Reference - changes WILL affect original
# df_ref = df_original         # dangerous!

# True copy - safe
df_safe = df_original.copy()
df_safe['Age'] = 999    # only changes df_safe

print('Original Age:', df_original['Age'].tolist())
print('Copy Age after modification:', df_safe['Age'].tolist())
print('→ Original is UNCHANGED ✅')

Original Age: [22.0, 38.0, 26.0, 35.0, 35.0]
Copy Age after modification: [999, 999, 999, 999, 999]
→ Original is UNCHANGED ✅


In [121]:
# Best practice for filtered subsets
survivors_copy = df[df['Survived'] == 1].copy()  # always .copy() after filter
survivors_copy['Survival_label'] = 'Yes'
print('New column added to copy safely:')
survivors_copy[['Name', 'Survival_label']].head(3)

New column added to copy safely:


,Name,Survival_label
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Yes
2,"Heikkinen, Miss. Laina",Yes
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Yes


---
## 22.Full EDA Pipeline on Titanic <a name='interview'></a>

A real data analysis workflow combining everything learned.


In [122]:
# ════════════════════════════════════════════════════════════
# FULL EDA PIPELINE - Titanic Dataset
# ════════════════════════════════════════════════════════════

print('STEP 1: Load & Inspect')
print('-' * 40)
df_eda = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
print(f'Shape: {df_eda.shape}')
print(f'Columns: {df_eda.columns.tolist()}')

STEP 1: Load & Inspect
----------------------------------------
Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [123]:
print('STEP 2: Missing Value Summary')
print('-' * 40)
missing = pd.DataFrame({
    'Missing Count': df_eda.isnull().sum(),
    'Missing %':     (df_eda.isnull().sum() / len(df_eda) * 100).round(1)
})
missing[missing['Missing Count'] > 0]

STEP 2: Missing Value Summary
----------------------------------------


,Missing Count,Missing %
Age,177,19.9
Cabin,687,77.1
Embarked,2,0.2


In [124]:
print('STEP 3: Clean & Feature Engineer')
print('-' * 40)

df_eda['Age']      = df_eda['Age'].fillna(df_eda['Age'].median())
df_eda['Embarked'] = df_eda['Embarked'].fillna(df_eda['Embarked'].mode()[0])
df_eda['Has_Cabin']   = df_eda['Cabin'].notnull().astype(int)
df_eda['FamilySize']  = df_eda['SibSp'] + df_eda['Parch'] + 1
df_eda['Is_Alone']    = (df_eda['FamilySize'] == 1).astype(int)
df_eda['Age_group']   = df_eda['Age'].apply(lambda x: 'Child' if x < 18 else ('Senior' if x >= 60 else 'Adult'))
df_eda['Fare_group']  = df_eda['Fare'].apply(lambda f: 'Budget' if f < 10 else ('Standard' if f < 50 else 'Premium'))

print('New columns added:', ['Has_Cabin', 'FamilySize', 'Is_Alone', 'Age_group', 'Fare_group'])
print('Remaining NaNs:', df_eda[['Age', 'Embarked', 'Cabin']].isnull().sum().to_dict())

STEP 3: Clean & Feature Engineer
----------------------------------------
New columns added: ['Has_Cabin', 'FamilySize', 'Is_Alone', 'Age_group', 'Fare_group']
Remaining NaNs: {'Age': 0, 'Embarked': 0, 'Cabin': 687}


In [125]:
print('STEP 4: Survival Analysis')
print('-' * 40)
overall = df_eda['Survived'].mean()
print(f'Overall survival rate: {overall:.1%}')
print()
print('By Sex:')
print(df_eda.groupby('Sex')['Survived'].mean().round(3))
print()
print('By Pclass:')
print(df_eda.groupby('Pclass')['Survived'].mean().round(3))
print()
print('By Age_group:')
print(df_eda.groupby('Age_group')['Survived'].mean().round(3))

STEP 4: Survival Analysis
----------------------------------------
Overall survival rate: 38.4%

By Sex:
Sex
female    0.742
male      0.189
Name: Survived, dtype: float64

By Pclass:
Pclass
1    0.630
2    0.473
3    0.242
Name: Survived, dtype: float64

By Age_group:
Age_group
Adult     0.364
Child     0.540
Senior    0.269
Name: Survived, dtype: float64


In [126]:
print('STEP 5: Correlation with Survival')
print('-' * 40)
num_cols = df_eda.select_dtypes(include='number')
corr = num_cols.corr()['Survived'].sort_values(ascending=False)
print(corr.round(3))

STEP 5: Correlation with Survival
----------------------------------------
Survived       1.000
Has_Cabin      0.317
Fare           0.257
Parch          0.082
FamilySize     0.017
PassengerId   -0.005
SibSp         -0.035
Age           -0.065
Is_Alone      -0.203
Pclass        -0.338
Name: Survived, dtype: float64


In [127]:
print('STEP 6: Key Findings')
print('-' * 40)

# Women survived more
women_surv = df_eda[df_eda['Sex'] == 'female']['Survived'].mean()
men_surv   = df_eda[df_eda['Sex'] == 'male']['Survived'].mean()
print(f'Women survival: {women_surv:.1%}')
print(f'Men survival:   {men_surv:.1%}')

# 1st class survived most
for cls in [1, 2, 3]:
    rate = df_eda[df_eda['Pclass'] == cls]['Survived'].mean()
    print(f'Pclass {cls} survival: {rate:.1%}')

# Most expensive fares (top 5)
print()
print('Top 5 fares paid:')
print(df_eda[['Name', 'Fare', 'Pclass', 'Survived']].nlargest(5, 'Fare')[['Name', 'Fare', 'Survived']].to_string())

STEP 6: Key Findings
----------------------------------------
Women survival: 74.2%
Men survival:   18.9%
Pclass 1 survival: 63.0%
Pclass 2 survival: 47.3%
Pclass 3 survival: 24.2%

Top 5 fares paid:
                                   Name      Fare  Survived
258                    Ward, Miss. Anna  512.3292         1
679  Cardeza, Mr. Thomas Drake Martinez  512.3292         1
737              Lesurer, Mr. Gustave J  512.3292         1
27       Fortune, Mr. Charles Alexander  263.0000         0
88           Fortune, Miss. Mabel Helen  263.0000         1


---
## Interview Q&A Cheatsheet

| Question | Answer |
|----------|--------|
| `loc` vs `iloc` difference? | `loc` = label-based (inclusive slicing); `iloc` = position-based (exclusive slicing) |
| Why use `.copy()`? | To avoid SettingWithCopyWarning and accidental modification of original df |
| `dropna(how='any')` vs `how='all'`? | `any` drops if ANY column has NaN; `all` drops only if ALL columns are NaN |
| What does `axis=0` mean? | Collapse rows → operate column-wise |
| `apply` vs vectorized ops? | Vectorized is faster; use `apply` for complex custom logic |
| How to check NaN quickly? | `df.isnull().sum()` - gives count per column |
| `value_counts()` vs `unique()`? | `value_counts()` = frequency of each value; `unique()` = just the distinct values |
| What is `inplace=True`? | Modifies the original DataFrame; no return value; use with caution |
| `duplicated(keep='first')` behavior? | Marks all duplicates as True EXCEPT first occurrence |
| Pearson correlation range? | -1 to +1; 0 = no linear relationship |

---

## GitHub Push Checklist
```
day12_pandas_dataframes/
├── df_basics.py          ← Creation, attributes, null detection, filtering
├── df_advanced.py        ← Sorting, NaN, apply, corr, nlargest, insert, copy
├── titanic.csv           ← Dataset (download from Kaggle)
└── Day12_Pandas_DataFrames_Complete.ipynb  ← This notebook
